<a href="https://colab.research.google.com/github/epotto/Light-Pollution-Senior-Research/blob/main/Automated_SQM_Moon_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- STEP 1: INSTALL LIBRARIES ---
# We no longer need 'openpyxl' for CSVs
!pip install skyfield pandas pytz ephem

import pandas as pd
import ephem
from skyfield.api import load, wgs84
from skyfield import almanac
from pytz import timezone
from google.colab import files
import io

# --- STEP 2: UPLOAD YOUR FILE ---
print("Click the button below to upload your CSV file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# --- STEP 3: SETUP ASTRONOMY ENGINE ---
print("Loading NASA Ephemeris Data...")
ts = load.timescale()
planets = load('de421.bsp')
earth, moon, sun = planets['earth'], planets['moon'], planets['sun']
mt_zone = timezone('America/Denver')
utc_zone = timezone('UTC')

def get_phase_name(angle_degrees):
    """Maps the 0-360 lunar angle to a human-readable name."""
    if angle_degrees < 1.0 or angle_degrees > 359.0: return "New Moon"
    if 1.0 <= angle_degrees < 89.0: return "Waxing Crescent"
    if 89.0 <= angle_degrees < 91.0: return "First Quarter"
    if 91.0 <= angle_degrees < 179.0: return "Waxing Gibbous"
    if 179.0 <= angle_degrees < 181.0: return "Full Moon"
    if 181.0 <= angle_degrees < 269.0: return "Waning Gibbous"
    if 269.0 <= angle_degrees < 271.0: return "Last Quarter"
    if 271.0 <= angle_degrees <= 359.0: return "Waning Crescent"
    return "Unknown"

def calculate_moon_metrics(row):
    try:
        if pd.isna(row['Date']) or pd.isna(row['Time']):
            return pd.Series([None, None, None, None, None])

        # Fuse Date and Time columns
        timestamp_str = f"{row['Date']} {row['Time']}"
        local_time = pd.to_datetime(timestamp_str)
        if local_time.tzinfo is None:
            local_time = mt_zone.localize(local_time)
        t = ts.from_datetime(local_time)

        # Coordinates
        lat, lon = float(row['Latitude']), float(row['Longitude'])

        # --- Skyfield Calculations (Alt, Az, Phase) ---
        observer = earth + wgs84.latlon(lat, lon, elevation_m=1373)
        astrometric = observer.at(t).observe(moon)
        alt, az, _ = astrometric.apparent().altaz()

        phase_angle = almanac.moon_phase(planets, t)
        phase_name = get_phase_name(phase_angle.degrees)
        fraction = astrometric.fraction_illuminated(sun)

        # --- Ephem Calculation (Magnitude) ---
        obs = ephem.Observer()
        obs.lat = str(lat)
        obs.lon = str(lon)
        # ephem requires UTC datetime without timezone info
        obs.date = local_time.astimezone(utc_zone).replace(tzinfo=None)
        ephem_moon = ephem.Moon(obs)
        moon_mag = round(ephem_moon.mag, 4)

        return pd.Series([round(alt.degrees, 4), round(az.degrees, 4), round(fraction, 4), phase_name, moon_mag])
    except:
        return pd.Series([None, None, None, None, None])

# --- STEP 4: PROCESS AND DOWNLOAD AS CSV ---
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"Processing {len(df)} rows...")

# Apply calculations
df[['Moon_Alt_Calc', 'Moon_Az_Calc', 'Moon_Phase_Pcnt', 'Moon_Phase_Name', 'Moon_Magnitude']] = df.apply(calculate_moon_metrics, axis=1)

# SAVE AS CSV
output_name = 'Research_Data_Final.csv'
df.to_csv(output_name, index=False)

print("-" * 30)
print("Processing Complete! Your browser will now prompt you to save the CSV file.")
files.download(output_name)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.4/370.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.7/235.7 kB 10.2 MB/s eta 0:00:00
Click the button below to upload your CSV file:


Saving SQM_Readings_For_R.csv to SQM_Readings_For_R.csv
Loading NASA Ephemeris Data...


[#################################] 100% de421.bsp


Processing 7074 rows...
------------------------------
Processing Complete! Your browser will now prompt you to save the CSV file.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>